In [ ]:
# For tips on running notebooks in Google Colab, see
# https://docs.pytorch.org/tutorials/beginner/colab
# %matplotlib inline

[Learn the Basics](intro.html) \|\| **Quickstart** \|\|
[Tensors](tensorqs_tutorial.html) \|\| [Datasets &
DataLoaders](data_tutorial.html) \|\|
[Transforms](transforms_tutorial.html) \|\| [Build
Model](buildmodel_tutorial.html) \|\|
[Autograd](autogradqs_tutorial.html) \|\|
[Optimization](optimization_tutorial.html) \|\| [Save & Load
Model](saveloadrun_tutorial.html)

Quickstart
==========

This section runs through the API for common tasks in machine learning.
Refer to the links in each section to dive deeper.

Working with data
-----------------

PyTorch has two [primitives to work with
data](https://pytorch.org/docs/stable/data.html):
`torch.utils.data.DataLoader` and `torch.utils.data.Dataset`. `Dataset`
stores the samples and their corresponding labels, and `DataLoader`
wraps an iterable around the `Dataset`.


In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

In [2]:
!nvidia-smi

Sun Jul 26 21:56:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 592.19                 Driver Version: 592.19         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5050 ...  WDDM  |   00000000:01:00.0  On |                  N/A |
| N/A   42C    P5             13W /   42W |     425MiB /   8151MiB |     16%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

PyTorch offers domain-specific libraries such as
[TorchText](https://pytorch.org/text/stable/index.html),
[TorchVision](https://pytorch.org/vision/stable/index.html), and
[TorchAudio](https://pytorch.org/audio/stable/index.html), all of which
include datasets. For this tutorial, we will be using a TorchVision
dataset.

The `torchvision.datasets` module contains `Dataset` objects for many
real-world vision data like CIFAR, COCO ([full list
here](https://pytorch.org/vision/stable/datasets.html)). In this
tutorial, we use the FashionMNIST dataset. Every TorchVision `Dataset`
includes two arguments: `transform` and `target_transform` to modify the
samples and labels respectively.


In [3]:
# Download training data from open datasets.
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    # 1. Converts from PIL to Torch compatible Image format and then converts to Float32 and Scales like min max scaling
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

# Download test data from open datasets.
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

100.0%
100.0%
100.0%
100.0%


We pass the `Dataset` as an argument to `DataLoader`. This wraps an
iterable over our dataset, and supports automatic batching, sampling,
shuffling and multiprocess data loading. Here we define a batch size of
64, i.e. each element in the dataloader iterable will return a batch of
64 features and labels.


In [4]:
batch_size = 64

# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


In [5]:
test_dataloader.dataset.data.shape

torch.Size([10000, 28, 28])

Read more about [loading data in PyTorch](data_tutorial.html).


------------------------------------------------------------------------


Creating Models
===============

To define a neural network in PyTorch, we create a class that inherits
from
[nn.Module](https://pytorch.org/docs/stable/generated/torch.nn.Module.html).
We define the layers of the network in the `__init__` function and
specify how data will pass through the network in the `forward`
function. To accelerate operations in the neural network, we move it to
the
[accelerator](https://pytorch.org/docs/stable/torch.html#accelerators)
such as CUDA, MPS, MTIA, or XPU. If the current accelerator is
available, we will use it. Otherwise, we use the CPU.


In [6]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

# Define model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten() # Step 1 -> Flattens the 28x28 Image to 784 length single dimenstion 
        self.linear_relu_stack = nn.Sequential(
            # 1st Param 28*28 is the input shape of the image, in this case it is 28*28 which is 784
            nn.Linear(28*28, 512), # linear -> Pytorch, Dense -> Tensorflow/Keras
            nn.ReLU(), # Activation function which introduces the non-linearity so that Model will be able to learn the complex curves instead of simple lines to separate
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device)
print(model)

Using cuda device
NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Read more about [building neural networks in
PyTorch](buildmodel_tutorial.html).


------------------------------------------------------------------------


Optimizing the Model Parameters
===============================

To train a model, we need a [loss
function](https://pytorch.org/docs/stable/nn.html#loss-functions) and an
[optimizer](https://pytorch.org/docs/stable/optim.html).


In [7]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

In a single training loop, the model makes predictions on the training
dataset (fed to it in batches), and backpropagates the prediction error
to adjust the model\'s parameters.


In [8]:
def train(dataloader, model, loss_fn, optimizer):

    """
    Initialization: size = len(dataloader.dataset) gets the total number of samples in the dataset, and model.train() sets the model to training mode (enabling features like Dropout or Batch Normalization).
    Batch Iteration: The for loop iterates through the dataloader, which provides batches of features (X) and labels (y). These are moved to the device (GPU or CPU).
    Forward Pass: pred = model(X) passes the images through the network to get predictions, and loss = loss_fn(pred, y) calculates how far off those predictions were from the actual labels.
    Backpropagation:
        loss.backward() calculates the gradient of the loss with respect to every model parameter.
        optimizer.step() updates the model's weights based on those gradients.
        optimizer.zero_grad() clears the gradients from the previous step so they don't accumulate.
    Logging: Every 100 batches, it prints the current loss and the progress (number of images processed) to help you monitor the training process.
    """
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

We also check the model\'s performance against the test dataset to
ensure it is learning.


In [9]:
def test(dataloader, model, loss_fn):
    """
    Setting the Mode: model.eval() is called to put the model in 'evaluation mode'. This is important because it disables specific behaviors like Dropout or Batch Normalization that are only meant for training.
    Disabling Gradients: The with torch.no_grad(): block tells PyTorch not to calculate or store gradients. Since we aren't updating the weights during testing, this saves a significant amount of memory and makes the computation faster.
    The Loop: It iterates through the test_dataloader. For each batch:
        It moves the images (X) and labels (y) to your device (GPU/CPU).
        It gets the model's predictions: pred = model(X).
        It accumulates the total loss using loss_fn(pred, y).item().
        It calculates accuracy by checking if the index of the highest prediction matches the actual label: (pred.argmax(1) == y).
    Final Metrics: After checking all batches, it divides the total loss by the number of batches to get the average loss, and divides the total number of correct predictions by the dataset size to get the accuracy percentage.
    Output: Finally, it prints the results so you can see if the model's accuracy is improving over time.
    """
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

The training process is conducted over several iterations (*epochs*).
During each epoch, the model learns parameters to make better
predictions. We print the model\'s accuracy and loss at each epoch;
we\'d like to see the accuracy increase and the loss decrease with every
epoch.


In [10]:
epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.307403  [   64/60000]
loss: 2.293304  [ 6464/60000]
loss: 2.270800  [12864/60000]
loss: 2.267780  [19264/60000]
loss: 2.255696  [25664/60000]
loss: 2.227138  [32064/60000]
loss: 2.235696  [38464/60000]
loss: 2.205405  [44864/60000]
loss: 2.204795  [51264/60000]
loss: 2.175568  [57664/60000]
Test Error: 
 Accuracy: 44.1%, Avg loss: 2.164702 

Epoch 2
-------------------------------
loss: 2.172742  [   64/60000]
loss: 2.162398  [ 6464/60000]
loss: 2.102574  [12864/60000]
loss: 2.124306  [19264/60000]
loss: 2.081260  [25664/60000]
loss: 2.018875  [32064/60000]
loss: 2.042149  [38464/60000]
loss: 1.968039  [44864/60000]
loss: 1.976981  [51264/60000]
loss: 1.903874  [57664/60000]
Test Error: 
 Accuracy: 57.1%, Avg loss: 1.899156 

Epoch 3
-------------------------------
loss: 1.921932  [   64/60000]
loss: 1.896765  [ 6464/60000]
loss: 1.777875  [12864/60000]
loss: 1.829381  [19264/60000]
loss: 1.727785  [25664/60000]
loss: 1.667491  [32064/600

Read more about [Training your model](optimization_tutorial.html).


------------------------------------------------------------------------


Saving Models
=============

A common way to save a model is to serialize the internal state
dictionary (containing the model parameters).


In [14]:
model.state_dict()

OrderedDict([('linear_relu_stack.0.weight',
              tensor([[-0.0247, -0.0212,  0.0246,  ..., -0.0060, -0.0232, -0.0140],
                      [ 0.0178, -0.0014, -0.0020,  ..., -0.0092, -0.0346,  0.0184],
                      [-0.0032,  0.0215,  0.0062,  ...,  0.0290, -0.0325, -0.0070],
                      ...,
                      [-0.0131,  0.0176,  0.0287,  ...,  0.0116, -0.0032, -0.0317],
                      [ 0.0012,  0.0122,  0.0227,  ..., -0.0190, -0.0265, -0.0106],
                      [ 0.0170, -0.0068,  0.0020,  ..., -0.0142, -0.0258, -0.0142]],
                     device='cuda:0')),
             ('linear_relu_stack.0.bias',
              tensor([ 2.7593e-02,  3.3262e-02,  3.3664e-02,  1.6059e-02, -3.0201e-02,
                      -3.0073e-02,  2.1107e-02,  2.3953e-02, -1.9515e-02, -2.4030e-03,
                       2.4448e-02,  2.2585e-02, -2.2468e-02,  2.9927e-02, -3.1792e-02,
                      -3.1119e-02,  2.1513e-02, -1.3224e-02, -2.7388e-02,  4.4300

In [11]:
torch.save(model.state_dict(), "model.pth")
print("Saved PyTorch Model State to model.pth")

Saved PyTorch Model State to model.pth


Loading Models
==============

The process for loading a model includes re-creating the model structure
and loading the state dictionary into it.


In [12]:
model = NeuralNetwork().to(device)
model.load_state_dict(torch.load("model.pth", weights_only=True))

<All keys matched successfully>

This model can now be used to make predictions.


In [13]:
classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]
# Switching on the eval model so that Torch wont update the gradients
model.eval()
x, y = test_data[0][0], test_data[0][1]
# As Gradients are not required while testing/inferencing we are initing with no_grad
with torch.no_grad():
    x = x.to(device)
    pred = model(x)
    predicted, actual = classes[pred[0].argmax(0)], classes[y]
    print(f'Predicted: "{predicted}", Actual: "{actual}"')

Predicted: "Ankle boot", Actual: "Ankle boot"


Read more about [Saving & Loading your
model](saveloadrun_tutorial.html).
